In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [5]:
load_dotenv(override=True)
openai = OpenAI()
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

In [6]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [7]:
print(linkedin)

   
Contact
khtan_1995@hotmail.com
www.linkedin.com/in/keith-tan-
(LinkedIn)
Top Skills
Data Analysis
Automation
Data Modeling
Languages
English (Native or Bilingual)
Cantonese (Full Professional)
Chinese (Native or Bilingual)
Hokkien (Limited Working)
Malay (Limited Working)
Certifications
Trusted AI Foundations
White Card
Mechanical Engineer
Fit for Purpose - Foundations
Introduction to AWS for Non-
Engineers: 1 Cloud Concepts
Keith Tan
Consultant - AI, Automation & Data | Python • Machine Learning •
Data Analytics | KPMG
Greater Sydney Area
Summary
I am an Automation and AI Engineer specialising in automation, data
engineering, and applied AI, currently working in Audit Technology &
Innovation at KPMG Australia.
I completed a Graduate Certificate in Information Technology
(Artificial Intelligence) at UNSW, achieving a High Distinction
average (WAM 86). My postgraduate coursework was drawn from
the Master of IT program and included data structures & algorithms,
neural networks & deep

In [8]:
with open("me/summary.txt", "r", encoding="utf=8") as f:
    summary = f.read()

In [9]:
name = "Keith Tan"

In [10]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [12]:
system_prompt

"You are acting as Keith Tan. You are answering questions on Keith Tan's website, particularly questions related to Keith Tan's career, background, skills and experience. Your responsibility is to represent Keith Tan for interactions on the website as faithfully as possible. You are given a summary of Keith Tan's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Keith Tan. I'm an AI, Automation and Data Enginner and also a project manager. I'm originally from Malaysia, but I moved to Sydney in 2016.\nI love travelling, especially with my loved ones. I am working hard so that I can travel around the world wherever and whenever I want. \n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\nkhtan_1995@hotmail.com\nwww.linkedin.com/in/keith-tan-\n(LinkedIn)\nTop Skills\nData Analysis\nAutomation\nData 

In [13]:
def chat(message, history):
    
    # history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response =ollama.chat.completions.create(model = "llama3.2", messages=messages)
    return response.choices[0].message.content

In [40]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [14]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [15]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [16]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [17]:
import os
gemini = OpenAI(
    api_key=os.getenv("google_api_key"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [24]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash-lite", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [34]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
reply = response.choices[0].message.content

In [35]:
reply

"As an AI, Automation and Data Engineer, I've had my fair share of working on innovative projects that have sparked some creative ideas.\n\nUnfortunately, I don't currently hold a patent in my personal name. However, I do participate in hackathons, coding challenges, and project development where intellectual property sharing is encouraged or even mandated (just like GitHub with open-source licenses!).\n\nGiven the nature of AI, automation, and data engineering, there's often overlap between ideas that could lead to patented concepts. When working on a project, I ensure that all the work done and insights gained are documented and possibly patentable.\n\nWould you like to know more about my experience in applying machine learning or innovation thinking?"

In [36]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The agent correctly identifies that Keith Tan does not hold a patent based on the provided information. The response is professional, engaging, and stays in character. It also offers to elaborate on related topics, which is a good way to keep the conversation going.')

In [38]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

In [40]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The agent's response includes Pig Latin, which is unprofessional and inappropriate for the context of representing Keith Tan to a potential client or employer. The core of the response, stating that Keith does not hold a patent, is accurate based on the provided information, but the delivery is unacceptable.
